# PE6201 · A2 · Live Model Battery (Problem A)

**这个 Notebook 只做一件事：用你被分配的那个模型，把整套评估案例跑一遍，得出你自己那一行 pass rate。**

跑完约需 **20–40 分钟**，费用约 **US$0.3**（若你分到 Anthropic haiku 档，约 US$2.8）。用自己的 key，自己付。

---

## 你要做的只有 3 步

1. 菜单 **`Runtime` → `Run all`**（或按 `Ctrl` + `F9`）。之后全部自动，你不用再点任何东西。
2. 第 ② 个格子会弹出两个输入框：
   - **`MODEL`** —— 组长告诉你跑哪个，就选哪个；
   - **`API_KEY`** —— 粘贴**你自己的** OpenRouter key（`sk-or-` 开头）。
3. 跑完后，把最后一块 `=== 抄这一块 ===` 的内容，连同下载下来的 `results.json` 一起发给组长。

> 第一次用 Colab？不用装任何软件、不用配置环境、不用懂代码。打开链接就能跑。

---

## ⚠️ 三条铁律（违反会让全组数据作废）

| # | 规则 | 为什么 |
|---|---|---|
| 1 | **不要改案例、不要改 prompt** | 全组必须用**完全相同**的案例集和 prompt，否则横向对比不成立 |
| 2 | **只有模型名允许不同** | 模型名是全组之间唯一被允许的变量 |
| 3 | **用你自己的 key** | 共用一个 key，账单和运行记录就分不清是谁跑的 |

## 💰 花费参考（自己的 key，自己付）

| 档位 | 你的模型 | 整套 33 条 约 |
|---|---|---|
| cheap | Llama / Qwen / Gemini flash / gpt-4o-mini | ≈ US$0.27 |
| mid | Anthropic haiku | ≈ US$2.76 |

> 课程总额度是 US$10。**千万别选 frontier 档**（一个人一整套 ≈ US$13.78，直接爆额度）。

In [ ]:
#@title ① 只改这一格：选模型 + 填你的 key { display-mode: "form" }
#
#   MODEL          : 从下拉里选。组长让你跑哪个就选哪个。
#   MODEL_OVERRIDE : 通常留空。只有当下拉里没有你要的模型时，
#                    才把准确的模型 ID 手输在这里（会覆盖上面的选择）。
#   API_KEY        : 粘贴你自己的 OpenRouter key，sk-or- 开头。
#
#   填完直接往下走，不需要点别的东西。
# =====================================================================

MODEL = "meta-llama/llama-3.1-8b-instruct" #@param ["meta-llama/llama-3.1-8b-instruct", "qwen/qwen-2.5-7b-instruct", "google/gemini-flash-1.5", "anthropic/claude-3.5-haiku", "openai/gpt-4o-mini"]
MODEL_OVERRIDE = "" #@param {type:"string"}
API_KEY = "" #@param {type:"string"}

# =====================================================================
import os
MODEL = (MODEL_OVERRIDE.strip() or MODEL.strip())
os.environ["OPENROUTER_API_KEY"] = API_KEY

print("模型 :", MODEL)
if API_KEY.startswith("sk-or-"):
    print("key  : 已填写 (…%s)" % API_KEY[-4:])
else:
    print("key  : !! 还没填，或不是 sk-or- 开头。请回到上面的输入框填写 !!")

In [ ]:
#@title ② 准备代码（自动，不用改。约 20 秒） { display-mode: "form" }
import os, re, sys, shutil, subprocess, importlib

REPO     = "https://github.com/didaralmrt-sudo/PE6201_A2_Group5"
WORK     = "/content/PE6201_A2_Group5"
SCAFFOLD = os.path.join(WORK, "A2_scaffold")

# 1) 拉取仓库（公开仓库，无需账号、无需 token）
if not os.path.isdir(WORK):
    subprocess.run(["git", "clone", "-q", REPO, WORK], check=True)
else:
    subprocess.run(["git", "-C", WORK, "pull", "-q"], check=False)
os.chdir(SCAFFOLD)
shutil.rmtree("__pycache__", ignore_errors=True)
print("代码位置 :", SCAFFOLD)

# 2) 把 config.py 切到 live，并写入你选的模型（只动这两行，其他一律不碰）
src = open("config.py", encoding="utf-8").read()
src = re.sub(r'^BACKEND\s*=\s*"scripted"', 'BACKEND = "live"', src, count=1, flags=re.M)
src = re.sub(r'^MODEL\s*=\s*".*?"', 'MODEL = "%s"' % MODEL, src, count=1, flags=re.M)
open("config.py", "w", encoding="utf-8").write(src)
shutil.rmtree("__pycache__", ignore_errors=True)   # config.py 自己警告过这个坑

# 3) 心跳：每跑完一个案例打印一行，免得你以为死机了
open("_colab_run.py", "w", encoding="utf-8").write('''
import sys, time
import harness

_orig = harness.run_case
def run_case(cid, **kw):
    t0 = time.time()
    rec = _orig(cid, **kw)
    print("    %-11s turns=%-2s %-10s %6.1fs  $%.4f"
          % (cid, rec.get("turns"), rec.get("stopped_by"),
             time.time() - t0, rec.get("cost_usd", 0)), flush=True)
    return rec

harness.run_case = run_case   # run_set() 在 harness 内部调用它，所以补丁生效
import run_eval
sys.exit(run_eval.main(["run_eval.py"] + sys.argv[1:]))
''')

# 4) 确认配置真的生效了
import config
importlib.reload(config)
print("BACKEND  :", config.BACKEND)
print("MODEL    :", config.MODEL)
print("案例数据 :", config.data_root())


def run(args):
    # 跑 run_eval.py，输出实时刷出来（不然 30 分钟一片安静，你会以为卡死）
    env = dict(os.environ, OPENROUTER_API_KEY=API_KEY)
    p = subprocess.Popen([sys.executable, "-u", "_colab_run.py"] + list(args),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, env=env, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    return p.returncode

In [ ]:
#@title ③ 先用 1 个案例试跑（约 1 分钟，≈ US$0.01）—— 验证 key 和模型能用 { display-mode: "form" }
#
#   目的：如果 key 填错了、或这个模型不支持结构化输出，在这里就报错，
#         不至于白等 30 分钟。
#   看到 "DECISION RECORD" 和 "CODE CHECK PASS/FAIL" 就算通过。
# =====================================================================
import json

first = json.load(open("../A2_reference_data/data_A/claims.json",
                       encoding="utf-8"))[0]["claim_id"]
print("试跑案例 :", first)
print("模型     :", MODEL)
print("-" * 68)

code = run([first])

print("-" * 68)
if code == 0:
    print("✅ 试跑通过。继续跑第 ④ 格。")
else:
    print("❌ 试跑失败。把上面的红色报错截图发给组长，先别往下跑。")

In [ ]:
#@title ④ 跑完整套（20–40 分钟）—— 中途不要关页面 { display-mode: "form" }
#
#   跑的过程中：每个案例跑完会打印一行，安静是正常的，别关页面。
#   Colab 免费版最长可连续跑 12 小时，足够了。
# =====================================================================
print("开始跑整套评估集。中途安静是正常的，每跑完一个案例会多一行。\n")
code = run(["--all"])
print("\nEXIT =", code)
print("（EXIT = 0 表示正常跑完）")

In [ ]:
#@title ⑤ 抄给组长：这一块 + results.json { display-mode: "form" }
import json, datetime
d = json.load(open("results.json", encoding="utf-8"))
s, res = d["summary"], d["results"]
tin  = sum(r["record"].get("tokens_in",  0) for r in res)
tout = sum(r["record"].get("tokens_out", 0) for r in res)

print("=" * 62)
print("=== 抄这一块，发给组长 ===")
print("model        : %s" % MODEL)
print("date         : %s" % datetime.date.today().isoformat())
print("backend      : live")
print("trials       : %s" % s["trials"])
print("passed       : %s" % s["passed"])
print("pass_rate    : %.1f%%" % (100.0 * s["pass_rate"]))
print("median_turns : %s" % s["median_turns"])
print("tokens_in    : %s" % tin)
print("tokens_out   : %s" % tout)
print("cost_usd     : $%.4f" % s["cost_usd"])
print("=" * 62)
print()
print("还差一步：跑下面第 ⑥ 格，把 results.json 下载下来一并发给组长。")

In [ ]:
#@title ⑥ 下载 results.json（发给组长） { display-mode: "form" }
try:
    from google.colab import files
    files.download("results.json")
    print("已开始下载 results.json —— 把它连同上面那块文字一起发给组长。")
except Exception as e:
    print("自动下载没成功（%r）。" % e)
    print("手动拿：左侧文件夹图标 → content/PE6201_A2_Group5/A2_scaffold/")
    print("        → 右键 results.json → Download")